# Reconnaissance et analyse automatique d'expressions faciales

**Fondamentaux du Deep Learning** - Master, Semestre 1 (2026-2027)

**Binome :** Mon Watts ([@damonwatts11](https://github.com/damonwatts11)) - Huet ([@SpectreAH](https://github.com/SpectreAH))  
**Formatrice :** Hanane Zerdoum - **Soutenance :** vendredi 9 octobre 2026

---

## Plan du notebook

| Partie | Contenu | Statut |
|:------:|---------|--------|
| 0 | Configuration et imports | - |
| 1 | Recherche, comprehension et preparation des donnees | Obligatoire |
| 2 | Modele de reference (reseau dense) | Obligatoire |
| 3 | Construction d'un CNN avec Keras | Obligatoire |
| 4 | Entrainement du reseau | Obligatoire |
| 5 | Evaluation et analyse des erreurs | Obligatoire |
| 6 | Experimentation et amelioration (>= 3 experiences) | Obligatoire |
| 7 | Enrichissement avec les nouvelles notions | Enrichissement |
| 8 | Detection de plusieurs visages (YOLO) | Extension |
| 9 | Extension a la video | Extension (bonus) |


## 0 - Configuration et imports

Imports communs, graine aleatoire (reproductibilite) et constantes globales du projet.
Sur Google Colab, activez le GPU : *Execution -> Modifier le type d'execution -> GPU*.

In [ ]:
# Decommenter sur Colab si besoin :
# !pip install -q ultralytics seaborn

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

print('TensorFlow', tf.__version__)
print('GPU disponible :', tf.config.list_physical_devices('GPU'))

In [ ]:
# --- Reproductibilite ---
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# --- Constantes du projet (a ajuster selon le dataset choisi) ---
IMG_SIZE    = (48, 48)   # taille d'entree du reseau
CHANNELS    = 1          # 1 = niveaux de gris, 3 = couleur
BATCH_SIZE  = 64
EPOCHS      = 30

# Classes (exemple a 7 expressions - a adapter au dataset)
CLASS_NAMES = ['colere', 'degout', 'peur', 'joie', 'tristesse', 'surprise', 'neutre']
NUM_CLASSES = len(CLASS_NAMES)

## 1 - Recherche, comprehension et preparation des donnees

**Choix du dataset** (fait partie du travail). Pistes : FER-2013, FERPlus, CK+, AffectNet, RAF-DB.

### A presenter
- source (lien, auteurs, **licence**)
- nombre total d'images
- classes et nombre d'images **par classe** (desequilibre ?)
- dimensions et format (niveaux de gris / couleur, type de fichier)
- plusieurs exemples d'images par categorie

In [ ]:
# TODO : charger le dataset.
# Exemple (FER-2013 en CSV) ou via image_dataset_from_directory si dossiers par classe.
#
# DATA_DIR = 'data/'
# train_ds = keras.utils.image_dataset_from_directory(
#     DATA_DIR + 'train', labels='inferred', label_mode='categorical',
#     color_mode='grayscale' if CHANNELS == 1 else 'rgb',
#     image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED)

# X, y = ...  # images (N, H, W, C) et labels

In [ ]:
# --- Exploration : distribution des classes ---
# counts = pd.Series(y).value_counts().sort_index()
# counts.index = CLASS_NAMES
# counts.plot(kind='bar'); plt.title('Images par classe'); plt.show()

In [ ]:
# --- Affichage d'exemples par categorie ---
# fig, axes = plt.subplots(NUM_CLASSES, 5, figsize=(10, 2 * NUM_CLASSES))
# for i, cls in enumerate(CLASS_NAMES):
#     idx = np.where(y == i)[0][:5]
#     for j, k in enumerate(idx):
#         axes[i, j].imshow(X[k].squeeze(), cmap='gray'); axes[i, j].axis('off')
#     axes[i, 0].set_ylabel(cls)
# plt.show()

### Preparation des donnees

Role de chaque etape a expliquer : redimensionnement, **normalisation** des pixels, preparation/encodage des labels, decoupage **train / validation / test**.

> Le jeu de **test** est reserve a l'evaluation finale : il ne sert jamais de validation.

In [ ]:
# --- Normalisation + encodage + split ---
# X = X.astype('float32') / 255.0                       # normalisation [0, 1]
# y_cat = keras.utils.to_categorical(y, NUM_CLASSES)    # one-hot
#
# X_train, X_tmp, y_train, y_tmp = train_test_split(
#     X, y_cat, test_size=0.30, random_state=SEED, stratify=y)
# X_val, X_test, y_val, y_test = train_test_split(
#     X_tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=y_tmp.argmax(1))
#
# print(X_train.shape, X_val.shape, X_test.shape)

## 2 - Modele de reference (reseau dense)

Reseau simple servant de **point de comparaison** avec le CNN.  
`Image -> Flatten -> Dense + activation -> Dense -> Softmax`

A savoir expliquer : transformation de l'image en entree, poids et biais, propagation avant, fonctions d'activation, fonction de perte, retropropagation, descente de gradient, sortie multiclasse (Softmax).

In [ ]:
def build_baseline():
    model = keras.Sequential([
        keras.Input(shape=(*IMG_SIZE, CHANNELS)),
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dense(NUM_CLASSES, activation='softmax'),
    ], name='baseline_dense')
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

baseline = build_baseline()
baseline.summary()

## 3 - Construction d'un CNN avec Keras

Proposez **et justifiez** votre propre architecture (plusieurs blocs conv + pooling).  
`Conv2D+ReLU -> MaxPooling -> Conv2D+ReLU -> MaxPooling -> Flatten -> Dense -> Softmax`

A expliquer : filtres, convolution, feature maps, kernel, stride, padding, ReLU, pooling, Flatten, Dense, couche de sortie. Suivez l'evolution des **dimensions** et le **nombre de parametres** couche par couche (`model.summary()`).

In [ ]:
def build_cnn():
    model = keras.Sequential([
        keras.Input(shape=(*IMG_SIZE, CHANNELS)),
        layers.Conv2D(32, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation='softmax'),
    ], name='cnn')
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

cnn = build_cnn()
cnn.summary()

## 4 - Entrainement du reseau

Justifiez : fonction de perte, optimiseur, batch size, nombre d'epoques, metriques.  
Etudiez les courbes **loss / accuracy** (train vs validation) et identifiez un eventuel **surapprentissage**.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(patience=4, factor=0.5),
]

# history = cnn.fit(X_train, y_train,
#                   validation_data=(X_val, y_val),
#                   epochs=EPOCHS, batch_size=BATCH_SIZE,
#                   callbacks=callbacks)

In [ ]:
def plot_history(history):
    h = history.history
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(h['loss'], label='train'); ax[0].plot(h['val_loss'], label='val')
    ax[0].set_title('Loss'); ax[0].legend()
    ax[1].plot(h['accuracy'], label='train'); ax[1].plot(h['val_accuracy'], label='val')
    ax[1].set_title('Accuracy'); ax[1].legend()
    plt.show()

# plot_history(history)

## 5 - Evaluation et analyse des erreurs

L'accuracy seule ne suffit pas (classes desequilibrees). Produisez une **matrice de confusion** et analysez : expressions bien reconnues, souvent confondues, les plus difficiles. Affichez des exemples `image + vraie classe + classe predite + probabilite`, dont plusieurs **erreurs**.

In [ ]:
# --- Evaluation sur le test ---
# test_loss, test_acc = cnn.evaluate(X_test, y_test)
# y_pred = cnn.predict(X_test)
# y_pred_cls = y_pred.argmax(1); y_true_cls = y_test.argmax(1)
#
# print(classification_report(y_true_cls, y_pred_cls, target_names=CLASS_NAMES))
#
# cm = confusion_matrix(y_true_cls, y_pred_cls)
# sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
# plt.xlabel('Predit'); plt.ylabel('Vrai'); plt.show()

## 6 - Experimentation et amelioration

**Au minimum 3 experiences**, en ne modifiant qu'un nombre limite de parametres a la fois (couches, filtres, kernels, pooling, neurones, batch size, learning rate, optimiseur, epoques) ou techniques (Dropout, Data Augmentation, Early Stopping, regularisation).

Presentez un **tableau comparatif** et justifiez le modele final.

| # | Modification | Val. accuracy | Observations |
|:-:|--------------|:-------------:|--------------|
| 1 | _(baseline CNN)_ | | |
| 2 | | | |
| 3 | | | |

In [ ]:
# --- Exemple : Data Augmentation ---
# data_aug = keras.Sequential([
#     layers.RandomFlip('horizontal'),
#     layers.RandomRotation(0.1),
#     layers.RandomZoom(0.1),
# ])
# Inserer data_aug juste apres la couche Input du CNN pour l'experience correspondante.

## 7 - Enrichissement avec les nouvelles notions

Pistes : Data Augmentation, **Transfer Learning** (reseaux pre-entraines), regularisation, detection d'objets, YOLO.

In [ ]:
# --- Exemple : Transfer Learning (necessite CHANNELS = 3) ---
# base = keras.applications.MobileNetV2(
#     input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet')
# base.trainable = False
# tl = keras.Sequential([
#     base, layers.GlobalAveragePooling2D(),
#     layers.Dense(128, activation='relu'), layers.Dropout(0.3),
#     layers.Dense(NUM_CLASSES, activation='softmax')])
# tl.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

## 8 - Extension : detection de plusieurs visages (YOLO)

`Image -> detection des visages -> bounding boxes -> extraction -> CNN -> expressions`

**Attention** : un YOLO standard (COCO) detecte des *personnes*, pas des *visages*. Utilisez un modele entraine aux visages (ou justifiez une autre solution). Chaque visage extrait doit subir **exactement le meme pretraitement** que les images d'entrainement.

Notions : classification vs detection, bounding box, confidence score, IoU, NMS, modele pre-entraine, (fine-tuning), precision/recall/mAP.

In [ ]:
# from ultralytics import YOLO
# detector = YOLO('yolov8n-face.pt')  # modele de detection de visages
#
# def analyser_image(img):
#     results = detector(img)
#     for box in results[0].boxes:
#         x1, y1, x2, y2 = map(int, box.xyxy[0])
#         visage = img[y1:y2, x1:x2]
#         # MEME pretraitement que l'entrainement (resize, gris/couleur, /255)
#         # pred = cnn.predict(prep(visage))
#         # dessiner la box + CLASS_NAMES[pred.argmax()] + score
#     return img

## 9 - Extension a la video (bonus)

`Video -> images successives -> detection -> extraction -> CNN -> predictions -> affichage`

In [ ]:
# import cv2
# cap = cv2.VideoCapture('data/demo.mp4')
# while cap.isOpened():
#     ok, frame = cap.read()
#     if not ok: break
#     frame = analyser_image(frame)
#     # ecrire / afficher la frame annotee
# cap.release()

## Conclusion

Neurone -> MLP -> Backpropagation -> Keras -> CNN -> Convolution -> Classification multiclasse -> Softmax -> Amelioration -> Detection -> YOLO -> Application complete de vision par ordinateur.

**Livrables (soutenance 9 oct. 2026)** : notebook executable de bout en bout, presentation, demonstration du modele final.